In [1]:
import sys
from pathlib import Path
# Add src directory to path to import utils package
sys.path.insert(0, str(Path('..') / 'src'))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    ensure_nuclei_labels_output_dir,
    load_precomputed_nuclei_labels_if_available,
)
from utils.segmentation import (
    predict_nuclei_labels,
    simulate_fluo_from_bf,
    generate_rough_root_3d,
    fill_root_3d,
    smooth_outer_root_surface_3d
)
from utils.inference import predict_tiled_unet

import tifffile
import napari
import torch



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


2026-04-18 05:24:35,438 [INFO] WRITING LOG OUTPUT TO C:\Users\adiezsanchez\.cellpose\run.log
2026-04-18 05:24:35,440 [INFO] 
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0
2026-04-18 05:24:35,518 [INFO] ** TORCH CUDA version installed and working. **
2026-04-18 05:24:35,519 [INFO] ** TORCH CUDA version installed and working. **
2026-04-18 05:24:35,520 [INFO] >>>> using GPU (CUDA)
2026-04-18 05:24:36,753 [INFO] >>>> loading model C:\Users\adiezsanchez\.cellpose\models\cpsam


In [2]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Point to the local PanSeg model UNet3D weights and config files
# You can choose from lightsheet_3D_unet_root_ds1x, lightsheet_3D_unet_root_ds2x, lightsheet_3D_unet_root_ds3x, confocal_3D_unet_sa_meristem_cells & generic_confocal_3D_unet
MODEL_DIR = "../plantseg_models/lightsheet_3D_unet_root_ds3x"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor) 
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor) – FRET
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor) - used for nuclei segmentation
MARKERS = (("edCerulean_CTRL", 0), ("edCitrine_FRET", 1), ("edCitrine_CTRL", 2), ("brightfield", 3))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 0 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 2

In [3]:
# Iterate through the .lif files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

lif_containers

['..\\raw_data\\20260129_ABACUS timepoints_mock 3h.lif',
 '..\\raw_data\\20260129_ABACUS timepoints_sorb 3h.lif',
 '..\\raw_data\\20260204_Splitplate_mock_6h.lif',
 '..\\raw_data\\20260204_Splitplate_sorb_6h.lif']

In [4]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)

Image name: Col0 1, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (115, 4, 256, 1024)
Image name: Col0 2, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (116, 4, 256, 1024)
Image name: Col0 3, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (135, 4, 256, 1024)
Image name: Col0 4, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (125, 4, 256, 1024)
Image name: Col0 5, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (107, 4, 256, 1024)
Image name: Col0 6, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (114, 4, 256, 1024)
Image name: Col0 7, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (118, 4, 256, 1024)


In [5]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

2026-04-18 05:24:41,789 [INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


[<Image layer 'edCerulean_CTRL' at 0x1ab726fe7a0>,
 <Image layer 'edCitrine_FRET' at 0x1ab72c33400>,
 <Image layer 'edCitrine_CTRL' at 0x1ab72c57be0>,
 <Image layer 'brightfield' at 0x1ab72c8fca0>]

In [6]:
# Simulate fluorescently labelled cell walls from brightfield input image
sim_fluo_cell_walls = simulate_fluo_from_bf(lif_image, MARKERS)

# Add simulated fluorescently labelled plant cell boundaries to Napari viewer
viewer.add_image(sim_fluo_cell_walls,
                name="PanSeg_UNet3D_input",
                colormap="gray",
                blending="additive")


<Image layer 'PanSeg_UNet3D_input' at 0x1aace5ad090>

In [7]:
# Predict root cell boundary probability maps using a pre-trained UNet3D model
root_pmaps = predict_tiled_unet(
    raw=sim_fluo_cell_walls,
    input_layout="ZYX",
    model_dir=MODEL_DIR,
    patch=(80, 160, 160),
    patch_halo=(8, 16, 16),
    stride_ratio=0.75,
    batch_size=1,
    device="cuda" if torch.cuda.is_available() else "cpu",
    use_amp=True,
)

# root_pmaps: (C_out, Z, Y, X)
viewer.add_image(root_pmaps[0], name="root_unet_pmap", colormap="viridis", blending="additive")

c:\Users\adiezsanchez\githubrepos\saramorg_fret_nroot\notebooks\..\src\utils\inference.py:163: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(weights_path,

<Image layer 'root_unet_pmap' at 0x1ab733d92d0>

In [8]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_nuclei_labels_output_dir(RAW_DATA_DIRECTORY, lif_container_id)
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_nuclei_labels_if_available(nuclei_labels_dir, lif_image_name)

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)

# Add the prediction to the napari viewer
viewer.add_labels(nuclei_labels)

Nuclei labels directory: ..\raw_data\nuclei_labels\20260129_ABACUS timepoints_mock 3h
x_um: 0.7568359375, y_um: 0.75461640625, z_um: 0.399525
Rescale factor: 0.5286637076611433
Predictions already calculated for: Col0 3 ...loading


<Labels layer 'nuclei_labels' at 0x1ab844efd60>

In [9]:
#TODO: Add logic to create folder to store root_3D_mask, check if it exists and load, else compute (merge with nuclei_labels_check logic)

In [13]:
# Generate a rough 3D root mask
filled_3d_closed = generate_rough_root_3d(root_pmaps, nuclei_labels, probability_threshold=0.9, visualize=True)

# Fill internal gaps inside rough 3D root mask
filled_root_3d = fill_root_3d(filled_3d_closed, occupancy_threshold=0.9, slice_aware_filling=True, visualize=True)

# Smooth root outer surface to remove small protrusions
smooth_root_3d = smooth_outer_root_surface_3d(filled_root_3d, erosion=5, smoothing=3, visualize=True)

In [11]:
#TODO: Move to feature_extraction.py

import numpy as np
from scipy.ndimage import distance_transform_edt
from skimage.measure import regionprops

def calculate_distance_to_root_surface(nuclei_labels, root_3d_mask, visualize=False):
    """
    Calculate a normalized distance map from the root surface and assign a depth value to each nucleus label, based on its centroid.
    The normalized distance is defined as the distance from each point inside the root mask to the nearest root surface, divided by the maximum thickness.
    Optionally, display the result in the global Napari viewer.

    Args:
        nuclei_labels (np.ndarray): 3D label array of nuclei (Z, Y, X).
        root_3d_mask (np.ndarray): Boolean 3D mask (Z, Y, X) of the root body.
        visualize (bool, optional): If True, display the normalized depth map in Napari.

    Returns:
        np.ndarray: 3D array (same shape as input) with normalized per-nucleus depth values; zero outside labeled regions.
    """
    # Pad the root mask with a border of zeros to ensure boundary proximity is detected at the edge of the image.
    mask_padded = np.pad(root_3d_mask.astype(bool), pad_width=1, mode='constant', constant_values=0)

    # Compute the distance transform on the padded mask.
    dist_padded = distance_transform_edt(mask_padded)

    # Remove padding to restore the shape to original spatial dimensions.
    dist_map = dist_padded[1:-1, 1:-1, 1:-1]

    # Normalize distances by the maximum value inside the root (aproximate root radius).
    max_dist = dist_map.max()
    if max_dist == 0:
        raise ValueError("Distance map is empty. Check mask.")

    dist_norm = dist_map / max_dist

    # Compute depth for each nucleus label based on the centroid coordinates in the normalized distance map.
    depth_per_label = {}
    props = regionprops(nuclei_labels)
    for prop in props:
        label_id = prop.label
        z, y, x = map(int, prop.centroid)  # Centroid coordinates
        depth = dist_norm[z, y, x]
        depth_per_label[label_id] = depth

    # Create an image where each voxel in a nucleus gets the assigned depth value for its label.
    depth_image = np.zeros_like(dist_map, dtype=float)
    for label_id, depth in depth_per_label.items():
        depth_image[nuclei_labels == label_id] = depth

    # Enhance contrast (clip the depth values between 1st and 99th percentile and re-normalize).
    nonzero = depth_image > 0
    if np.any(nonzero):
        p1, p99 = np.percentile(depth_image[nonzero], (1, 99))
        depth_image = np.clip(depth_image, p1, p99)
    depth_image = (depth_image - depth_image.min()) / (
        depth_image.max() - depth_image.min() + 1e-8
    )

    # Display the resulting normalized per-nucleus depth in Napari if requested.
    if visualize:
        viewer.add_image(
            depth_image,
            name="nuclei_depth_normalized",
            colormap="viridis",
            blending="additive"
        )

    return depth_image

In [12]:
#TODO: Add logic to create folder to store depth map, check if it exists and load, else compute (merge with nuclei_labels_check logic)

# Calculate distance from each nuclei centroid to the root surface
# This will be used to approximate to which tissue layer each nucleus belongs
nuclei_depth_map = calculate_distance_to_root_surface(nuclei_labels, smooth_root_3d, visualize=True)